# 📄 Document Transformer — LayoutLM Fine-Tuning on SROIE

Trains a **LayoutLM** model for token-level NER on the **ICDAR-2019 SROIE** receipt dataset.

**Fields extracted:** `COMPANY`, `DATE`, `ADDRESS`, `TOTAL`

**Pipeline:**
1. Environment setup
2. Load & parse SROIE data
3. Tokenise and align BIO labels
4. Fine-tune LayoutLM (30 epochs, early stopping)
5. Evaluate with seqeval (precision / recall / F1)
6. Save best checkpoint

## 0 — Environment Setup

In [ ]:
# Uncomment to install if needed
# !pip install transformers datasets seqeval evaluate Pillow tqdm torch

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')
print(f'Python:       {sys.version}')

## 1 — Configuration

In [ ]:
from transformer.config import (
    BATCH_SIZE, DEVICE, EARLY_STOP_PATIENCE, EPOCHS, EVAL_STRATEGY,
    LABELS, LABEL2ID, ID2LABEL, LEARNING_RATE, LOGGING_STEPS, MAX_SEQ_LENGTH,
    METRIC_FOR_BEST, MODEL_NAME, NUM_LABELS, OUTPUT_DIR, RANDOM_SEED,
    SAVE_TOTAL_LIMIT, TRANSFORMER_CKPT, WARMUP_RATIO, WEIGHT_DECAY,
    DATA_DIR, BOX_DIR, KEY_DIR, IMG_DIR,
)

print('=' * 60)
print('  Document Transformer — Configuration Summary')
print('=' * 60)
print(f'  Device          : {DEVICE}')
print(f'  Base model      : {MODEL_NAME}')
print(f'  Epochs          : {EPOCHS}')
print(f'  Batch size      : {BATCH_SIZE}')
print(f'  Learning rate   : {LEARNING_RATE}')
print(f'  Max seq length  : {MAX_SEQ_LENGTH}')
print(f'  Num labels      : {NUM_LABELS}')
print(f'  Labels          : {LABELS}')
print(f'  Checkpoint dir  : {TRANSFORMER_CKPT}')
print('=' * 60)

## 2 — Load & Parse SROIE Dataset

In [ ]:
from transformer.dataset import load_sroie_data

print('[1/4] Loading and parsing SROIE dataset ...')
train_raw, val_raw = load_sroie_data()

print(f'  Train samples : {len(train_raw)}')
print(f'  Val   samples : {len(val_raw)}')

sample = train_raw[0]
print(f"--- Sample preview (id={sample['id']}) ---")
print(f"  Words  (first 10): {sample['words'][:10]}")
print(f"  Boxes  (first 10): {sample['bboxes'][:10]}")
print(f"  Tags   (first 10): {[LABELS[t] for t in sample['ner_tags'][:10]]}")

## 3 — Tokenise & Build HuggingFace Datasets

In [ ]:
from transformer.dataset import build_hf_datasets

print('Tokenising datasets (this may take a minute) ...')
train_ds, val_ds = build_hf_datasets()

print(f'  Train dataset : {train_ds}')
print(f'  Val   dataset : {val_ds}')

## 4 — Initialise LayoutLM Model

In [ ]:
from transformer.model import DocumentTransformer
from transformer.dataset import get_tokenizer

print('[2/4] Initialising LayoutLM model ...')
doc_transformer = DocumentTransformer.from_pretrained()
model     = doc_transformer.model
tokenizer = get_tokenizer()

print(f'  Model type    : {type(model).__name__}')
print(f'  Num labels    : {model.config.num_labels}')
print(f'  id2label      : {model.config.id2label}')

## 5 — Configure Trainer

In [ ]:
from transformers import (
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    TrainingArguments,
    Trainer,
)
from transformer.evaluate import compute_metrics

print('[3/4] Configuring Trainer ...')

training_args = TrainingArguments(
    output_dir                  = TRANSFORMER_CKPT,
    eval_strategy               = EVAL_STRATEGY,
    save_strategy               = EVAL_STRATEGY,
    learning_rate               = LEARNING_RATE,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    num_train_epochs            = EPOCHS,
    weight_decay                = WEIGHT_DECAY,
    warmup_steps                = int(WARMUP_RATIO * EPOCHS * 100),
    logging_steps               = LOGGING_STEPS,
    save_total_limit            = SAVE_TOTAL_LIMIT,
    load_best_model_at_end      = True,
    metric_for_best_model       = METRIC_FOR_BEST,
    greater_is_better           = True,
    seed                        = RANDOM_SEED,
    report_to                   = 'none',
    fp16                        = True,  # Enables mixed precision to save VRAM and avoid cuBLAS errors
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_ds,
    eval_dataset     = val_ds,
    processing_class = tokenizer,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
)

print(f'  Epochs              : {EPOCHS}')
print(f'  Steps/epoch (est.)  : {len(train_ds) // BATCH_SIZE}')
print(f'  Early stop patience : {EARLY_STOP_PATIENCE} epochs')
print('  Trainer ready ✓')

## 6 — Train

In [ ]:
print('[4/4] Starting training ...')
train_result = trainer.train()

print('\n' + '=' * 60)
print('  Training complete!')
print(f'  Total steps    : {train_result.global_step}')
print(f'  Train loss     : {train_result.training_loss:.4f}')
print('=' * 60)

## 7 — Save Best Model

In [ ]:
import os
best_model_dir = os.path.join(TRANSFORMER_CKPT, 'best_model')
trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)
print(f'  ✓  Best model saved → {best_model_dir}')

## 8 — Final Evaluation

In [ ]:
from transformer.evaluate import evaluate_checkpoint

metrics = evaluate_checkpoint(trainer, split='val')

print('--- Metrics Summary ---')
for k, v in sorted(metrics.items()):
    print(f'  {k:35s}: {v}')

## 9 — Save Training Summary JSON

In [ ]:
import json, os

summary = {
    'global_step':   train_result.global_step,
    'training_loss': train_result.training_loss,
    'final_metrics': metrics,
}

summary_path = os.path.join(OUTPUT_DIR, 'transformer_training_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'  ✓  Training summary saved → {summary_path}')
print('  Done! 🎉')

---
## 10 — (Optional) Quick Inference Test

In [ ]:
from transformer.model import DocumentTransformer
import torch

loaded = DocumentTransformer.load_checkpoint()

sample = val_raw[0]
words  = sample['words']
boxes  = sample['bboxes']

enc = tokenizer(
    words,
    is_split_into_words=True,
    truncation=True,
    padding='max_length',
    max_length=MAX_SEQ_LENGTH,
    return_tensors='pt',
)

word_ids    = enc.word_ids()
bbox_tensor = torch.tensor(
    [boxes[w] if w is not None else [0, 0, 0, 0] for w in word_ids]
).unsqueeze(0)

tokenized_input = {
    'input_ids':      enc['input_ids'],
    'attention_mask': enc['attention_mask'],
    'token_type_ids': enc['token_type_ids'],
    'bbox':           bbox_tensor,
}

extracted = loaded.extract_fields(tokenized_input, words, word_ids)

print(f"Sample ID: {sample['id']}")
print('Extracted fields:')
for field, value in extracted.items():
    print(f"  {field:10s}: {value if value else '(not detected)'}")